# K-ABENA — Niveau 1 : Classification Logistique
**K appliqué sur la perte BCE individuelle** — les observations bien classifiées (faible BCE) sont mineures.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from kabena_ml.integrations.sklearn_wrapper import KabenaWrapper

# Données
X, y = make_classification(n_samples=2000, n_features=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Standard
lr_std = LogisticRegression(max_iter=500).fit(X_train, y_train)

# K-ABENA — +1 wrapper, API identique
lr_ka = KabenaWrapper(
    LogisticRegression(max_iter=1),
    K=0.20, N=0.3, epochs=300, task="classification", verbose=True
).fit(X_train, y_train)

print("=== Standard ===")
print(classification_report(y_test, lr_std.predict(X_test)))
print("=== K-ABENA ===")
print(classification_report(y_test, lr_ka.predict(X_test)))
print(f"Gain moyen : {lr_ka.stats_['mean_gain_pct']:.1f}%")

## Données déséquilibrées — K-ABENA Stratifié

In [ ]:
from kabena_ml.integrations.sklearn_wrapper import KabenaWrapper

X_imb, y_imb = make_classification(n_samples=2000, weights=[0.85, 0.15], random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X_imb, y_imb, test_size=0.2, stratify=y_imb)

# KabenaWrapper avec stratified=True — filtrage indépendant par classe
ka_strat = KabenaWrapper(
    LogisticRegression(max_iter=1),
    K=0.20, N=0.3, epochs=300,
    task="classification", stratified=True
).fit(Xtr, ytr)

from sklearn.metrics import f1_score
print(f"F1 K-ABENA Stratifié : {f1_score(yte, ka_strat.predict(Xte), average='weighted'):.4f}")